# 🏥 MedMamba — Eye Image HB Estimation
**Repo:** [YubiaoYue/MedMamba](https://github.com/YubiaoYue/MedMamba)  
**Paper:** *MedMamba: Vision Mamba for Medical Image Classification* (2024)  
**Tasks:** ① Binary classification (Anemic vs Non-Anemic) ② HB regression

MedMamba is the first generalized Vision-Mamba for medical image classification.
We adapt its SS-Conv-SSM block for dual-task output.


In [ ]:
# ── 0. Setup ──────────────────────────────────────────────────────────────
import subprocess, sys
subprocess.run(["git", "clone", "--depth=1",
                "https://github.com/YubiaoYue/MedMamba.git", "MedMamba"], check=False)
sys.path.insert(0, "MedMamba")

pkgs = ["mamba-ssm", "causal-conv1d", "einops", "timm", "pandas", "openpyxl",
        "scikit-learn", "matplotlib", "tqdm"]
for p in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", p, "-q"], check=False)
print("✅ Done")


In [ ]:
# ── 1. Imports & Config ───────────────────────────────────────────────────
import os, sys, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_absolute_error
import matplotlib.pyplot as plt
from tqdm import tqdm
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

CFG = dict(
    image_dir  = "/kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/left_eye",
    csv_path   = "/kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/merge_excel_1.xlsx",
    image_col  = "Patient ID",
    hb_col     = "HB",
    hb_thresh  = 12.0,
    img_size   = 224,
    epochs     = 30,
    batch_size = 8,
    lr         = 2e-4,
    cls_weight = 1.0,
    reg_weight = 0.5,
    val_split  = 0.2,
    seed       = 42,
)


In [ ]:
# ── 2. Load MedMamba & add dual heads ────────────────────────────────────
try:
    from MedMamba import VSSM  # Original MedMamba backbone
    print("✅ Loaded MedMamba VSSM backbone")
    BASE_AVAILABLE = True
except Exception as e:
    print(f"Could not load MedMamba directly: {e}")
    BASE_AVAILABLE = False

# Wrapper that adds regression head regardless of backbone availability
class MedMambaDualHead(nn.Module):
    def __init__(self, num_classes=2, embed_dim=96, depths=[2,2,9,2],
                 use_original=True):
        super().__init__()
        self.use_original = use_original and BASE_AVAILABLE
        if self.use_original:
            self.backbone = VSSM(num_classes=num_classes,
                                  depths=depths, dims=[embed_dim*2**i for i in range(4)])
            feat_dim = embed_dim * 2**(len(depths)-1)
            # Patch the final head to be identity
            self.backbone.classifier = nn.Identity()
        else:
            # Lightweight fallback CNN+SSM
            from torchvision.models import efficientnet_b0
            base = efficientnet_b0(weights=None)
            feat_dim = 1280
            base.classifier = nn.Identity()
            self.backbone = base
            print("Using EfficientNet-B0 fallback backbone")

        self.cls_head = nn.Linear(feat_dim, num_classes)
        self.reg_head = nn.Sequential(
            nn.Linear(feat_dim, feat_dim//4), nn.GELU(), nn.Linear(feat_dim//4, 1))

    def forward(self, x):
        feat = self.backbone(x)
        if feat.dim() > 2:
            feat = feat.mean([-2,-1])
        return self.cls_head(feat), self.reg_head(feat)

model = MedMambaDualHead(num_classes=2, embed_dim=96).to(DEVICE)
print(f"Params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M")


In [ ]:
# ── 3. Dataset (same as Notebook 1) ──────────────────────────────────────
import os
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split
import pandas as pd

class EyeHBDataset(Dataset):
    def __init__(self, df, image_dir, image_col, hb_col, hb_thresh, transform=None):
        self.df=df.reset_index(drop=True); self.image_dir=image_dir
        self.image_col=image_col; self.hb_col=hb_col
        self.hb_thresh=hb_thresh; self.transform=transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]; pid=str(row[self.image_col]); hb=float(row[self.hb_col])
        label=0 if hb < self.hb_thresh else 1
        for ext in [".jpg",".jpeg",".png",""]:
            p=os.path.join(self.image_dir, pid+ext)
            if os.path.exists(p): break
        img=Image.open(p).convert("RGB")
        if self.transform: img=self.transform(img)
        return img, torch.tensor(label,dtype=torch.long), torch.tensor([[hb]],dtype=torch.float32)

T_train = transforms.Compose([transforms.Resize((256,256)), transforms.RandomCrop(224),
            transforms.RandomHorizontalFlip(), transforms.ColorJitter(0.2,0.2,0.1),
            transforms.ToTensor(), transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
T_val   = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(),
            transforms.Normalize([.485,.456,.406],[.229,.224,.225])])

df = pd.read_excel(CFG["csv_path"])
train_df, val_df = train_test_split(df, test_size=CFG["val_split"], random_state=CFG["seed"],
                    stratify=(df[CFG["hb_col"]]<CFG["hb_thresh"]).astype(int))
train_loader = DataLoader(EyeHBDataset(train_df, CFG["image_dir"], CFG["image_col"],
                           CFG["hb_col"],CFG["hb_thresh"],T_train),
                           batch_size=CFG["batch_size"],shuffle=True,num_workers=2)
val_loader   = DataLoader(EyeHBDataset(val_df, CFG["image_dir"], CFG["image_col"],
                           CFG["hb_col"],CFG["hb_thresh"],T_val),
                           batch_size=CFG["batch_size"],shuffle=False,num_workers=2)
print(f"Train: {len(train_df)} | Val: {len(val_df)}")


In [ ]:
# ── 4. Train (copy-paste from Notebook 1 — same training loop) ───────────
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["epochs"])
ce_loss   = nn.CrossEntropyLoss(); mse_loss = nn.MSELoss()

def compute_loss(logits, hb_pred, labels, hb_true):
    return CFG["cls_weight"]*ce_loss(logits,labels) + CFG["reg_weight"]*mse_loss(hb_pred,hb_true)

best = float("inf")
for epoch in range(1, CFG["epochs"]+1):
    model.train(); tl=0
    for imgs,labels,hb in tqdm(train_loader, desc=f"Ep{epoch}", leave=False):
        imgs,labels,hb=imgs.to(DEVICE),labels.to(DEVICE),hb.to(DEVICE)
        optimizer.zero_grad()
        l,r=model(imgs); loss=compute_loss(l,r,labels,hb)
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.); optimizer.step()
        tl+=loss.item()
    scheduler.step(); tl/=len(train_loader)
    model.eval(); vl=correct=total=0; hbp=[]; hbt=[]
    with torch.no_grad():
        for imgs,labels,hb in val_loader:
            imgs,labels,hb=imgs.to(DEVICE),labels.to(DEVICE),hb.to(DEVICE)
            l,r=model(imgs); vl+=compute_loss(l,r,labels,hb).item()
            correct+=(l.argmax(1)==labels).sum().item(); total+=labels.size(0)
            hbp+=r.cpu().squeeze().tolist(); hbt+=hb.cpu().squeeze().tolist()
    vl/=len(val_loader)
    if vl<best: best=vl; torch.save(model.state_dict(),"best_medmamba.pth")
    if epoch%5==0 or epoch==1:
        print(f"Ep{epoch:3d} | TL:{tl:.4f} | VL:{vl:.4f} | Acc:{correct/total:.3f} | MAE:{mean_absolute_error(hbt,hbp):.2f}")
